In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("healthcare_cleaned.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace(" ", "_")
df["Date_of_Admission"] = pd.to_datetime(df["Date_of_Admission"])
df["Discharge_Date"]    = pd.to_datetime(df["Discharge_Date"])
print(df.shape)    # expect (54966, 16)

(54966, 19)


In [3]:
df.head()

,Name,Age,Gender,Blood_Type,Medical_Condition,Date_of_Admission,Doctor,Hospital,Insurance_Provider,Billing_Amount,Room_Number,Admission_Type,Discharge_Date,Medication,Test_Results,Length_of_Stay,Admission_Year,Admission_Month,Age_Group
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.28131,328,Urgent,2024-02-02,Paracetamol,Normal,2,2024,2024-01,19-30
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.32729,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,2019,2019-08,61-75
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.09608,205,Emergency,2022-10-07,Aspirin,Normal,15,2022,2022-09,76+
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78241,450,Elective,2020-12-18,Ibuprofen,Abnormal,30,2020,2020-11,19-30
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.31781,458,Urgent,2022-10-09,Penicillin,Abnormal,20,2022,2022-09,31-45


#### cross-check the aggregates match

In [5]:
print("Number of rows:", len(df))
print("Missing values:", df.isna().sum().sum())
print("Negative bills:", (df["Billing_Amount"] < 0).sum())

Number of rows: 54966
Missing values: 0
Negative bills: 106


In [6]:
## missing value is zero

#### Remove repeated row

In [8]:
same_visit = ["Name", "Gender", "Blood_Type", "Medical_Condition", "Date_of_Admission",
              "Doctor", "Hospital", "Insurance_Provider", "Billing_Amount", "Room_Number",
              "Admission_Type", "Discharge_Date", "Medication", "Test_Results"]   # every column except Age

print("Repeated rows:", df.duplicated(subset=same_visit).sum())

clean = df.drop_duplicates(subset=same_visit)
print("Rows left:", len(clean))

Repeated rows: 4966
Rows left: 50000


#### How many patient per conditions?

In [10]:
clean["Medical_Condition"].value_counts()

Medical_Condition
Arthritis       8439
Diabetes        8384
Hypertension    8319
Cancer          8294
Obesity         8292
Asthma          8272
Name: count, dtype: int64

#### Average bill and average stay per condition?

In [12]:
clean.groupby("Medical_Condition")[["Billing_Amount", "Length_of_Stay"]].mean().round(1)

,Billing_Amount,Length_of_Stay
Medical_Condition,,
Arthritis,25486.2,15.6
Asthma,25681.2,15.7
Cancer,25234.7,15.5
Diabetes,25654.5,15.4
Hypertension,25494.2,15.4
Obesity,25784.0,15.5


#### Test results per condition (in %)

In [14]:
(pd.crosstab(clean["Medical_Condition"], clean["Test_Results"], normalize="index") * 100).round(1)

Test_Results,Abnormal,Inconclusive,Normal
Medical_Condition,,,
Arthritis,34.5,33.0,32.5
Asthma,32.6,32.9,34.5
Cancer,33.7,33.2,33.1
Diabetes,33.8,33.0,33.2
Hypertension,32.7,33.5,33.8
Obesity,34.0,32.9,33.1


#### Do the numbers move together?

In [16]:
clean[["Age", "Billing_Amount", "Length_of_Stay", "Room_Number"]].corr().round(2)

,Age,Billing_Amount,Length_of_Stay,Room_Number
Age,1.00,-0.0,0.01,-0.0
Billing_Amount,-0.00,1.0,-0.00,-0.0
Length_of_Stay,0.01,-0.0,1.00,-0.0
Room_Number,-0.00,-0.0,-0.00,1.0


#### Patients per Year?

In [18]:
clean["Date_of_Admission"] = pd.to_datetime(clean["Date_of_Admission"])
clean.groupby(clean["Date_of_Admission"].dt.year).size()

C:\Users\shubh\AppData\Local\Temp\ipykernel_54028\451553139.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean["Date_of_Admission"] = pd.to_datetime(clean["Date_of_Admission"])


Date_of_Admission
2019     6635
2020    10156
2021     9883
2022     9936
2023     9929
2024     3461
dtype: int64

In [19]:
df.groupby("Medical_Condition").agg(
    patients=("Name", "count"),
    avg_billing=("Billing_Amount", "mean"),
    avg_los=("Length_of_Stay", "mean")
).round(2).sort_values("patients", ascending=False)

,patients,avg_billing,avg_los
Medical_Condition,,,
Arthritis,9218,25511.78,15.50
Diabetes,9216,25660.48,15.43
Hypertension,9151,25503.06,15.44
Obesity,9146,25804.36,15.45
Cancer,9140,25152.32,15.50
Asthma,9095,25633.46,15.68


#### Correlation matrix

In [21]:
df[["Age", "Billing_Amount", "Length_of_Stay", "Room_Number"]].corr().round(3)

,Age,Billing_Amount,Length_of_Stay,Room_Number
Age,1.000,-0.003,0.008,-0.000
Billing_Amount,-0.003,1.000,-0.005,-0.004
Length_of_Stay,0.008,-0.005,1.000,-0.005
Room_Number,-0.000,-0.004,-0.005,1.000


### What we found
- After removing repeats, there are about **50,000** real visits.
- Every condition has almost the same bill, stay, and test result split.
- All correlations are close to 0.
- Admissions per year are steady.
- **Conclusion:** this data has no real patterns. It is made-up (synthetic) data, so the value of this project is the clean process you followed, not a prediction.